# Разбор A. Исследовательский вопрос

В предыдущем уроке мы отделили объект от его описания. Теперь лаборатория спрашивает, можно ли выбирать растворитель по прогнозу растворимости. У истрадефиллина нет в этом наборе сопоставимого измерения: обещать точность для него нельзя. Вспомогательная ESOL позволяет сформулировать опровержимую гипотезу о методе. До вычисления предположите, будет ли ошибка одинаковой у хорошо и плохо растворимых соединений. Для B вернитесь к аторвастатину: определите форму, pH, температуру, равновесную или кинетическую растворимость, а не переносите числа ESOL на лекарство.

Прочитайте закреплённый CSV и сохраните row_id. Не сортируйте прогнозы отдельно от наблюдений. Рассчитайте остатки, сравните средние ошибки для y < −3 и y ≥ −3. Постройте наблюдение–прогноз и диагональ. Затем выпишите до следующего опыта endpoint, критерий ошибки и правила исключения.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M01-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
rows=esol()
errors=[r['predicted']-r['measured'] for r in rows]
mae=sum(abs(e) for e in errors)/len(errors)
rmse=math.sqrt(sum(e*e for e in errors)/len(errors))
print({'n':len(rows),'MAE_logS':mae,'RMSE_logS':rmse})


{'n': 1128, 'MAE_logS': 0.6978617021276595, 'RMSE_logS': 0.9101322725549837}


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
for label,group in [('low',[r for r in rows if r['measured'] < -3]),('high',[r for r in rows if r['measured'] >= -3])]:
    print(label,len(group),sum(abs(r['predicted']-r['measured']) for r in group)/len(group))
fig,ax=plt.subplots()
ax.scatter([r['measured'] for r in rows],[r['predicted'] for r in rows],s=7,alpha=.4)
ax.plot([-12,3],[-12,3],color='black');ax.set(xlabel='Observed logS',ylabel='Predicted logS')
publish_plot(fig,'esol.png')


low 543 0.744340699815838
high 585 0.6547196581196582


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'n':len(rows),'mae_logS':mae,'rmse_logS':rmse,'kind':'calculated','target_drug_prediction':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M01-L02','A',result,ROOT,inputs=['practicuum/shared/data/esol.csv'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\practicuum\evidence\20261119-main-unification\notebooks\workspace\my_research\evidence\M01-L02\20261119T123225494742Z-A-d2c98e86b8f84698ba7f56563f7b6e79.json


Расположение относительно диагонали показывает направление ошибки, но не причину. Корреляция не гарантирует калибровку. Для решения об истрадефиллине нужен новый релевантный набор измерений. Вопрос для перехода: каким образом другой исследователь воспроизведёт выбор строк и версии расчёта?

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?

# M01. Как наставник читает первичную работу

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Crevar-Sakač et al. (2013). An Improved HPLC Method with the Aid of a Chemometric Protocol: Simultaneous Determination of Atorvastatin and Its Metabolites in Plasma**. [Локальный полный текст](../../../../../shared/data/source-responses/hplc_article.xml), [DOI](https://doi.org/10.3390/molecules18032469). Откройте **Abstract, Introduction и назначение метода**.

Выберите предложение о назначении аналитического метода; отделите объект анализа, матрицу и измеряемый результат.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Можно ли из описания метода анализа плазмы заключить, что вещество избирательно накапливается в печени?**